In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from tqdm import tqdm

In [2]:
class XrayDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []

        self.classes = ["NORMAL", "PNEUMONIA"]
        self.class_to_idx = {"NORMAL": 0, "PNEUMONIA": 1}

        for cls in self.classes:
            folder = os.path.join(root_dir, cls)
            for img_name in os.listdir(folder):
                path = os.path.join(folder, img_name)
                self.samples.append((path, self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("L")  

        if self.transform:
            img = self.transform(img)

        return img, label

In [3]:
IMG_SIZE = 224
BATCH_SIZE = 64

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

test_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

train_dir = "Data/chest_xray/train"
val_dir   = "Data/chest_xray/val"
test_dir  = "Data/chest_xray/test"

train_ds = XrayDataset(train_dir, train_tf)
val_ds   = XrayDataset(val_dir, test_tf)
test_ds  = XrayDataset(test_dir, test_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

In [4]:
class PneumoniaCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1,1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [5]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device).float().unsqueeze(1)

        outputs = model(imgs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = (torch.sigmoid(outputs) >= 0.5).int()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), accuracy_score(all_labels, all_preds)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    probs = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device).float().unsqueeze(1)

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            p = torch.sigmoid(outputs)
            preds = (p >= 0.5).int()

            probs.extend(p.cpu().numpy().flatten())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return {
        "loss": total_loss / len(loader),
        "acc": accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds),
        "recall": recall_score(all_labels, all_preds),
        "f1": f1_score(all_labels, all_preds),
        "auc": roc_auc_score(all_labels, probs)
    }

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)

Running on: cuda


In [8]:
model = PneumoniaCNN().to(device)

In [9]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
EPOCHS = 17
best_f1 = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_stats = evaluate(model, val_loader, criterion, device)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}  | Train Acc: {train_acc:.4f}")
    print("Val:", val_stats)

    # ============= SAVE COMPLETE MODEL =============
    if val_stats["f1"] > best_f1:
        best_f1 = val_stats["f1"]

        torch.save({
            "model": model,                
            "epoch": epoch,
            "optimizer": optimizer.state_dict(),
            "f1": best_f1
        }, "pneumonia_model.pth")

        print("Model Saved")


Running on: cpu

Epoch 1/17
Train Loss: 0.5836  | Train Acc: 0.7429
Val: {'loss': 0.8554795384407043, 'acc': 0.5, 'precision': 0.5, 'recall': 1.0, 'f1': 0.6666666666666666, 'auc': 0.765625}
Model Saved

Epoch 2/17
Train Loss: 0.5646  | Train Acc: 0.7429
Val: {'loss': 0.8228709697723389, 'acc': 0.5, 'precision': 0.5, 'recall': 1.0, 'f1': 0.6666666666666666, 'auc': 0.796875}

Epoch 3/17
Train Loss: 0.5559  | Train Acc: 0.7429
Val: {'loss': 0.7233722805976868, 'acc': 0.5, 'precision': 0.5, 'recall': 1.0, 'f1': 0.6666666666666666, 'auc': 0.875}

Epoch 4/17
Train Loss: 0.4589  | Train Acc: 0.7487
Val: {'loss': 0.8517598509788513, 'acc': 0.5625, 'precision': 0.5333333333333333, 'recall': 1.0, 'f1': 0.6956521739130435, 'auc': 0.78125}
Model Saved

Epoch 5/17
Train Loss: 0.3949  | Train Acc: 0.7770
Val: {'loss': 1.246227741241455, 'acc': 0.5625, 'precision': 0.5333333333333333, 'recall': 1.0, 'f1': 0.6956521739130435, 'auc': 0.78125}

Epoch 6/17
Train Loss: 0.3689  | Train Acc: 0.8012
Val: {'l

In [10]:
checkpoint = torch.load("Models/pneumonia_model.pth", map_location=device,weights_only=False)

model = checkpoint["model"]
model = model.to(device)
model.eval()

test_stats = evaluate(model, test_loader, criterion, device)

print("\n===== TEST RESULTS =====")
print(test_stats)



===== TEST RESULTS =====
{'loss': 0.4365075409412384, 'acc': 0.8349358974358975, 'precision': 0.8253968253968254, 'recall': 0.9333333333333333, 'f1': 0.8760529482551144, 'auc': 0.8910694718387027}


In [11]:
def predict_image(img_path):
    model.eval()

    img = Image.open(img_path).convert("L")

    tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ])

    x = tf(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logit = model(x)
        prob = torch.sigmoid(logit).item()

    label = "PNEUMONIA" if prob >= 0.5 else "NORMAL"
    return prob, label